# Teaching LFM3-VL-2B to Transcribe Formulas into LaTeX with TRL (GRPO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/finetuning/notebooks/lfm3_vl_grpo_with_trl.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-Liquid4All%2Fcookbook-181717?logo=github)](https://github.com/Liquid4All/cookbook) [![Discord](https://img.shields.io/discord/1385439864920739850?color=7289da&label=Discord&logo=discord&logoColor=white)](https://discord.com/invite/liquid-ai)

This notebook trains [LFM3-VL-2B](https://huggingface.co/LiquidAI/LFM3-VL-2B) to turn pictures of math formulas into LaTeX, with Group Relative Policy Optimization (GRPO) from [TRL](https://github.com/huggingface/trl) and LoRA. We measure the transcriptions before and after training on formulas the model never saw.

GRPO needs no example answers to imitate, only a way to score the model's own answers. The model writes several answers to each picture, and it learns from the ones that score better than the others. LaTeX is a good fit: we can check the format of an answer and compare it with the reference automatically, and there are many ways to reason toward the right one.

## 🎯 What you'll find

- **The task**: a picture of a formula goes in, its LaTeX in a single `$$ ... $$` block comes out
- **Two rewards**: one for the format of the answer, one for how close the LaTeX is to the reference
- **Thinking kept on**: the model reasons before it answers, and GRPO also keeps its reasoning short
- **Evaluation**: format rate, exact match, similarity, and output length, on held-out formulas

# 📦 Installation & Setup

We pin every library to the versions this notebook was tested with.

In [ ]:
!pip install -qqq "torch==2.14.1" "torchvision==0.29.1" "torchao==0.18.0" "trl==1.16.0" "transformers==5.20.2" "peft==0.21.2" "accelerate==1.15.0" "datasets==5.1.0" "huggingface-hub==2.3.0" "tokenizers==0.23.3" "safetensors==0.8.0" --progress-bar off

In [ ]:
import os
import re
import torch
import transformers
import trl
import peft

os.environ["WANDB_DISABLED"] = "true"

print(f"📦 PyTorch version: {torch.__version__}")
print(f"🤗 Transformers version: {transformers.__version__}")
print(f"📊 TRL version: {trl.__version__}")
print(f"🎛️ PEFT version: {peft.__version__}")

# LFM3-VL-2B is released in bfloat16. GPUs without bfloat16 support, like the T4, load it in float16 instead:
# the conversion is exact for almost every weight, and the LoRA adapters train in float32 either way.
use_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.bfloat16 if use_bf16 else torch.float16
print(f"🔢 Precision: {dtype}")

# Loading the model from Transformers 🤗

LFM3-VL-2B ships its modeling code with the checkpoint, so we load it with `trust_remote_code=True`. We pad on the left, so that batched generation continues each prompt right where it ends.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model_id = "LiquidAI/LFM3-VL-2B"

print("📚 Loading processor...")
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
processor.tokenizer.padding_side = "left"

print("🧠 Loading model...")
model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=dtype, device_map="auto", trust_remote_code=True)

print("✅ Model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")

# 📚 Load the formula dataset

[LaTeX_OCR](https://huggingface.co/datasets/unsloth/LaTeX_OCR) holds about 76,000 rendered formulas from physics papers, each with the LaTeX that produced it. The reference LaTeX puts a space between all tokens (`\frac { 1 } { 2 }`) and uses the conventions of these papers: `{ \cal R }` where recent LaTeX writes `\mathcal { R }`, extra braces like `{ \dot { \phi } } ^ { 2 }`. The original model reads the formulas well but writes its own style, so GRPO has something to learn.

GRPO learns from the model's own answers, so a few hundred pictures are enough: we train on 240 formulas and evaluate on 100 others.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("unsloth/LaTeX_OCR").rename_column("text", "latex")
train_dataset = dataset["train"].shuffle(seed=42).select(range(240))
eval_dataset = dataset["test"].shuffle(seed=42).select(range(100))
print(dataset)

example = train_dataset[0]
print(f"\n🎯 Reference LaTeX:\n{example['latex']}")
example["image"]

## Build the prompts

Each prompt is one user message with the picture and the instruction. GRPO only needs prompts: the reference LaTeX stays in its own column, and TRL passes it to the reward functions. TRL also puts the picture of the `image` column in front of the text.

We keep LFM3-VL's reasoning on, as the chat template does by default: the model thinks, then answers.

In [ ]:
INSTRUCTION = "Transcribe the formula in the picture into LaTeX. Answer with the LaTeX only, in a single $$ ... $$ block."

def to_prompt(example):
    return {"prompt": [{"role": "user", "content": INSTRUCTION}]}

train_dataset = train_dataset.map(to_prompt)
print(train_dataset)

# 🏆 Define the rewards

TRL parses each completion into a message with the reasoning (`reasoning_content`) and the answer (`content`). A completion that runs out of tokens before the end of its reasoning has no answer, and gets no reward.

- **Format** (0 or 1): the answer is exactly one `$$ ... $$` block, with nothing around it.
- **LaTeX** (0 to 1): how close the LaTeX in the block is to the reference. We split both into LaTeX tokens (`\frac`, `{`, `x`, ...), so spaces don't count, and compute `1 - edit distance / length`. An exact transcription scores 1.

Both rewards also serve as evaluation metrics, with the exact match rate and the output length.

In [ ]:
BLOCK = re.compile(r"\s*\$\$([^$]+)\$\$\s*")  # <- the LaTeX inside holds no "$"
TOKEN = re.compile(r"\\[a-zA-Z]+|\\.|\S")

def latex_of(answer):
    """The LaTeX in the answer's first $$ ... $$ block, or None."""
    match = BLOCK.search(answer or "")
    return match.group(1) if match else None

def similarity(prediction, reference):
    """1 - (edit distance between the LaTeX token sequences) / (length of the longer one)."""
    a, b = TOKEN.findall(prediction), TOKEN.findall(reference)
    previous = list(range(len(b) + 1))
    for i, token_a in enumerate(a, 1):
        current = [i]
        for j, token_b in enumerate(b, 1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (token_a != token_b)))
        previous = current
    return 1 - previous[-1] / max(len(a), len(b), 1)

def format_reward(completions, **kwargs):
    return [float(BLOCK.fullmatch(completion[0].get("content") or "") is not None) for completion in completions]

def latex_reward(completions, latex, **kwargs):
    rewards = []
    for completion, reference in zip(completions, latex):
        prediction = latex_of(completion[0].get("content"))
        rewards.append(similarity(prediction, reference) if prediction is not None else 0.0)
    return rewards

reference = "\\frac { 1 } { 2 } m v ^ { 2 }"
for answer in ["$$\\frac{1}{2} m v^{2}$$", "$$\\frac{1}{2} m v^2$$", "The formula is $$\\frac{1}{2}mv^{2}$$", None]:
    completion = [{"role": "assistant", "content": answer}]
    print(f"{answer!r:<42} format {format_reward([completion])[0]:.0f} | LaTeX {latex_reward([completion], [reference])[0]:.2f}")

# 📏 Evaluate the model before training

We transcribe the 100 evaluation formulas with the sampling settings recommended in the model card (they are the checkpoint's generation defaults), and report:

- **Format**: the share of answers that are exactly one `$$ ... $$` block.
- **Exact match**: the share of transcriptions with the same LaTeX tokens as the reference.
- **Similarity**: the average LaTeX reward.
- **Output tokens**: the average length of the reasoning and the answer, which sets the latency.

In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def transcribe(model, images, max_new_tokens=2048):
    """The parsed message (reasoning and answer) of the model for each picture, and its length in tokens."""
    messages = [[{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": INSTRUCTION}]}] for image in images]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt", processor_kwargs={"padding": True}
    ).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)[:, inputs["input_ids"].shape[1]:]
    results = []
    for prompt_ids, output_ids in zip(inputs["input_ids"], outputs):
        output_ids = output_ids[output_ids != processor.tokenizer.pad_token_id].tolist()
        prompt_ids = prompt_ids[prompt_ids != processor.tokenizer.pad_token_id].tolist()
        results.append((processor.tokenizer.parse_response(output_ids, prefix=prompt_ids), len(output_ids)))
    return results

def evaluate(model, batch_size=50):
    model.eval()
    formats, exact, similarities, tokens = 0, 0, 0.0, 0
    for start in tqdm(range(0, len(eval_dataset), batch_size), desc="Evaluating"):
        batch = eval_dataset.select(range(start, min(start + batch_size, len(eval_dataset))))
        for (message, length), reference in zip(transcribe(model, batch["image"]), batch["latex"]):
            completion = [message]
            formats += format_reward([completion])[0]
            reward = latex_reward([completion], [reference])[0]
            similarities += reward
            exact += reward == 1.0
            tokens += length
    n = len(eval_dataset)
    return {"format": formats / n, "exact_match": exact / n, "similarity": similarities / n, "tokens": tokens / n}

def show(r, title):
    print(f"{title}: format {r['format']:.1%} | exact match {r['exact_match']:.1%} | similarity {r['similarity']:.3f} | {r['tokens']:.0f} output tokens")

torch.manual_seed(0)
baseline = evaluate(model)
show(baseline, "📏 Before training")

# 🎯 GRPO training (LoRA)

## Wrap the model with PEFT

In LFM3's language model, we adapt the attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`), the gated short convolutions (`in_proj`, `out_proj`), and the MLP (`w1`, `w2`, `w3`). LFM3's attention layers also have a per-head output gate, `gate_proj`; adding it to the targets is a reasonable choice. The vision encoder uses some of the same names, so we select the language model's layers with a regular expression and leave the vision encoder frozen.

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # optional: add "|gate_proj" to the group for LFM3's per-head attention gate
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|in_proj|out_proj|w1|w2|w3)",
    task_type="CAUSAL_LM",
)

## Launch training

At each step, the model writes 8 completions for each of 4 pictures. GRPO compares the rewards of the 8 completions of a picture: it makes the better ones more likely and the worse ones less likely.

We sample with more randomness than the model card recommends for inference (`temperature=1.0` instead of 0.1): GRPO only learns when the completions of a picture differ. And we limit completions to 1,024 tokens: a completion that is still reasoning at the limit gets no reward, so the model learns to keep its reasoning short.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="./LFM3-VL-2B-latex",
    max_steps=60,  # <- 60 steps × 4 pictures = one pass over the 240 training formulas
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,  # <- 32 completions per step: 4 pictures × 8 completions
    num_generations=8,
    max_completion_length=1024,
    temperature=1.0,
    learning_rate=5e-5,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=0.05,  # <- a float is a fraction of the total steps
    gradient_checkpointing=True,
    logging_steps=5,
    save_strategy="no",
    bf16=use_bf16,
    fp16=not use_bf16,  # <- float16 mixed precision with loss scaling on GPUs without bfloat16
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward, latex_reward],
    args=grpo_config,
    train_dataset=train_dataset,
    processing_class=processor,
    peft_config=peft_config,
)
trainer.model.print_trainable_parameters()

print("🚀 Starting GRPO training...")
trainer.train()
print("🎉 GRPO training completed!")
print(f"🧮 Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## Training curves

The rewards should go up and the completions should get shorter.

In [ ]:
import matplotlib.pyplot as plt

logs = [entry for entry in trainer.state.log_history if "reward" in entry]
steps = [entry["step"] for entry in logs]
figure, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for axis, key, title in zip(axes, ["rewards/format_reward/mean", "rewards/latex_reward/mean", "completions/mean_length"], ["Format reward", "LaTeX reward", "Completion length (tokens)"]):
    axis.plot(steps, [entry[key] for entry in logs])
    axis.set_title(title)
    axis.set_xlabel("step")
plt.tight_layout()
plt.show()

# 📊 Evaluate the trained model

In [ ]:
torch.manual_seed(0)
trained = evaluate(trainer.model)
show(baseline, "📏 Before training")
show(trained, "📏 After training")

# 🔍 Original vs trained model

We transcribe a few evaluation formulas with the original model and the trained one, and render the three LaTeX strings. With LoRA, we don't need to load the original model again: `disable_adapter()` temporarily turns the adapter off.

In [ ]:
from IPython.display import Markdown, display

trainer.model.eval()
for example in eval_dataset.select(range(4)):
    with trainer.model.disable_adapter():
        [(original, original_length)] = transcribe(trainer.model, [example["image"]])
    [(trained_message, trained_length)] = transcribe(trainer.model, [example["image"]])
    display(example["image"])
    for name, message, length in [("Original", original, original_length), ("Trained", trained_message, trained_length)]:
        prediction = latex_of(message.get("content"))
        reward = latex_reward([[message]], [example["latex"]])[0]
        print(f"{name} ({length} tokens, LaTeX reward {reward:.2f}): {prediction}")
    print(f"Reference: {example['latex']}")
    display(Markdown(f"Reference: $${example['latex']}$$"))

And the summary over the evaluation formulas:

In [ ]:
print(f"{'':<16}{'original':>10}{'trained':>10}")
for metric in ["format", "exact_match"]:
    print(f"{metric:<16}{baseline[metric]:>10.1%}{trained[metric]:>10.1%}")
print(f"{'similarity':<16}{baseline['similarity']:>10.3f}{trained['similarity']:>10.3f}")
print(f"{'output tokens':<16}{baseline['tokens']:>10.0f}{trained['tokens']:>10.0f}")

# 💾 Save and merge the model

We save the LoRA adapter, then merge it into the base weights. `save_pretrained` also copies LFM3-VL's modeling code next to the weights, so the merged model loads with `trust_remote_code=True` like the original one.

In [ ]:
trainer.save_model("./LFM3-VL-2B-latex-lora")

output_dir = "./LFM3-VL-2B-latex-merged"
merged_model = trainer.model.merge_and_unload()
merged_model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)
print(f"💾 Merged model saved to: {output_dir}")

# To push the merged model to the Hugging Face Hub, uncomment:
# merged_model.push_to_hub("your-username/LFM3-VL-2B-latex")
# processor.push_to_hub("your-username/LFM3-VL-2B-latex")

# 🚀 Next steps

- **Your own checker**: any program that scores an answer is a reward: a LaTeX compiler, a unit test, a schema validator, a calculator.
- **Longer training**: the dataset has 68,000 training formulas; more steps keep improving the rewards.
- **Faster generation**: most of the training time goes into writing completions. With an inference engine that supports LFM3-VL, set `use_vllm=True` in `GRPOConfig`.